# Theme 2: RAG Application with Gemma — Problem 2B: Open-Source Contributor Guide

| | |
|---|---|
| **Problem Statement** | **2B: Open-Source Contributor Guide** |
| **Model** | Google open-weight **Gemma 4** (`gemma-4-26b-a4b-it` / `gemma-4-31b-it`) |
| **Embedding Model** | `sentence-transformers/all-MiniLM-L6-v2` |
| **Platform** | Google Colab (CPU runtime, no GPU required) & Google AI Studio API |
| **Key Extensions** | Multi-document ingestion (.md, .txt, .pdf), section-aware chunking, source citations, relevance threshold refusal guardrail, 18-question evaluation suite, and "Where Do I Start?" onboarding wizard. |

---

## Table of Contents
1. [Step 0: Installation (run once)](#scrollTo=step-0)
2. [Step 1: Settings & Hyperparameters](#scrollTo=step-1)
3. [Step 2: Connect to Gemma 4 and Load Embedder](#scrollTo=step-2)
4. [Step 3: Ingest Contributor Documentation Corpus](#scrollTo=step-3)
5. [Step 4: Advanced Multi-Document RAG Pipeline](#scrollTo=step-4)
6. [Step 5: Bonus Features (Onboarding Wizard & RAG vs No-RAG)](#scrollTo=step-5)
7. [Step 6: Automated Evaluation & Ablation Benchmark (18 Questions)](#scrollTo=step-6)
8. [Step 7: Launch Interactive Web Application (Gradio)](#scrollTo=step-7)


<a name="step-0"></a>
## Step 0: Installation (run once)
Installs required packages in the Google Colab environment.


In [ ]:
# Installs required libraries (takes ~45 seconds)
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai


<a name="step-1"></a>
## Step 1: Settings & Hyperparameters
Configure Gemma 4 model, chunking parameters, retrieval depth (Top-K), and the relevance threshold for declining out-of-domain questions.


In [ ]:
# MODEL SELECTION
# "gemma-4-26b-a4b-it": Mixture of Experts (26B total / ~4B active). Fast and token-efficient. (Recommended default)
# "gemma-4-31b-it": Dense 31B parameters. Maximum capability.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# RAG CHUNKING PARAMETERS
# 120 words balances granular retrieval with complete context preservation for commands.
CHUNK_SIZE = 120
CHUNK_OVERLAP = 30  # 25% overlap prevents splitting critical instructions or code blocks
TOP_K = 3           # Number of most relevant passages retrieved for Gemma

# GUARDRAILS & RELEVANCE THRESHOLD
# If the highest cosine similarity score among retrieved passages is below this threshold,
# the question is considered out-of-domain and declined without calling Gemma.
SIMILARITY_THRESHOLD = 0.28
TEMPERATURE = 0.2   # Low temperature keeps answers factual and strictly faithful to documentation

# EMBEDDING MODEL
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# SYSTEM PROMPT INSTRUCTIONS FOR GEMMA 4
ANSWER_RULES = """You are an expert Open-Source Contributor Assistant. Answer the contributor's question using ONLY the provided documentation passages.
Rules:
- Each passage begins with its metadata: [Document: <filename>, Section: <section_title>].
- After each factual statement, cite the document and section in brackets, for example: (CONTRIBUTING.md, Section: Pull Request Workflow).
- If the answer is not contained in the context, DECLINE to answer and reply EXACTLY:
  "The requested information could not be found in the provided contributor documentation. For assistance with this topic, please check the project's GitHub Discussions or ask in the community Discord channel."
- Do NOT invent or hallucinate commands, URLs, or rules not present in the context.
- Keep answers concise, clear, and actionable (2 to 5 sentences)."""

print("✅ Settings configured successfully.")


<a name="step-2"></a>
## Step 2: Connect to Gemma and Load Embedder
Connects to Google AI Studio via `google.genai` using your free API key, and loads the `all-MiniLM-L6-v2` embedding model.


In [ ]:
import os
import numpy as np
from google import genai
from google.genai import types, errors

# Read API Key from Colab Secrets or Environment
def _get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)

def generate_gemma(user_text, system_text=ANSWER_RULES, temperature=TEMPERATURE, demo_reply="(Demonstration response)"):
    """Invokes Gemma 4 via Google GenAI SDK with minimal thinking latency."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model=GEMMA_MODEL,
            contents=user_text,
            config=types.GenerateContentConfig(
                system_instruction=system_text,
                temperature=temperature,
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return f"Error from Google AI Studio API: {e}"
    except Exception as e:
        return f"Error: {e}"

if DEMO_MODE:
    print("⚠️ Demonstration mode: No GOOGLE_API_KEY found in Colab Secrets.")
    print("Retrieval & refusal will work normally. Set GOOGLE_API_KEY in Secrets for live Gemma 4 answers.")
else:
    print(f"✅ Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    test_reply = generate_gemma("In one short sentence, what is an open-source contributor?")
    print(f"Gemma 4 Test: {test_reply}")

# Load Embedding Model
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print(f"✅ Loaded Embedding Model: {EMBED_MODEL_NAME}")


<a name="step-3"></a>
## Step 3: Ingest Contributor Documentation Corpus
We bundle realistic, high-quality contributor documentation files for a major open-source Python framework (FastAPI).
The corpus spans:
1. `README.md`: Project overview, features, community links, and license.
2. `CONTRIBUTING.md`: Contribution avenues, finding good first issues, pull request checklist.
3. `DEVELOPMENT_SETUP.md`: Python versions, virtual environment, dependency installation, pytest commands.
4. `STYLE_GUIDE.md`: Ruff formatting, 88-char limit, type annotations, pre-commit hooks.
5. `CODE_OF_CONDUCT.md`: Contributor Covenant v2.0, reporting contact, zero harassment policy.


In [ ]:
import os

DOCS_CORPUS = {
    "README.md": """# FastAPI Contributor Guide & Project Overview

Welcome to the FastAPI open-source project repository! FastAPI is a modern, fast, web framework for building APIs with Python 3.8+ based on standard Python type hints.

## Key Features
- Fast: Very high performance, on par with NodeJS and Go (thanks to Starlette and Pydantic).
- Intuitive: Great editor support. Completion everywhere. Less time debugging.
- Standards-based: Based on and fully compatible with OpenAPI and JSON Schema.

## Community & Getting Help
- GitHub Discussions: For general questions and beginner help, open a discussion at https://github.com/fastapi/fastapi/discussions.
- Discord Chat: Join our community Discord server at https://discord.gg/fastapi in the #contributors channel.

## License
FastAPI is licensed under the terms of the MIT license. See LICENSE for details.
""",

    "CONTRIBUTING.md": """# Contributing to FastAPI

Thank you for your interest in contributing to FastAPI! Contributions from developers of all skill levels are welcome.

## Ways to Contribute
You do not have to write code to contribute. You can:
1. Help others with questions on GitHub Discussions and Discord.
2. Translate documentation into your native language (check docs/<lang>).
3. Report bugs with minimal reproducible examples.
4. Fix open issues tagged with 'good first issue' or 'help wanted'.
5. Improve existing documentation and fix typos.

## Finding an Issue to Work On
- Start by checking issues labeled with 'good first issue' on GitHub.
- Comment on the issue asking to be assigned before starting work.
- If you have an idea for a new feature, open a GitHub Discussion first.

## Pull Request Workflow
1. Fork the repository on GitHub to your personal account.
2. Clone your fork locally to your machine.
3. Create a descriptive feature branch from master (e.g., git checkout -b fix/issue-1234-query-param).
4. Make your changes: Keep changes focused and atomic.
5. Write tests: Every bug fix or feature must include new or updated tests under tests/.
6. Run the test suite locally: bash scripts/test.sh or pytest.
7. Format and lint your code: bash scripts/format.sh and bash scripts/lint.sh.
8. Push to your fork and submit a Pull Request to the master branch.
9. PR Title: Use conventional commit style (e.g., fix: resolve query parameter type coercion). Reference issue (Fixes #1234).
10. Review Process: At least one core maintainer approval is required.
""",

    "DEVELOPMENT_SETUP.md": """# Development Environment Setup Guide

This guide walks you through setting up a complete local development environment.

## Prerequisites
- Python: Python 3.8, 3.9, 3.10, 3.11, or 3.12 installed.
- Git: Installed and configured with your name and email.
- Package Manager: We recommend pdm or standard Python venv with pip.

## Step-by-Step Installation
1. Clone the Repository:
   git clone https://github.com/<your-username>/fastapi.git
   cd fastapi
   git remote add upstream https://github.com/fastapi/fastapi.git

2. Create and Activate Virtual Environment:
   python -m venv .venv
   source .venv/bin/activate  # On Windows: .\\.venv\\Scripts\\Activate.ps1

3. Install Dependencies:
   pip install -e ".[all,dev,test]"
   # Or using PDM:
   pdm install --dev

4. Run the Test Suite:
   pytest
   # With coverage:
   pytest --cov=fastapi --cov-report=term-missing
   # Single test file:
   pytest tests/test_tutorial/test_first_steps.py

## Running Documentation Locally
FastAPI documentation uses mkdocs-material. To preview docs live with hot reloading:
mkdocs serve
Then visit http://127.0.0.1:8000 in your browser.
""",

    "STYLE_GUIDE.md": """# FastAPI Code Style & Quality Standards

## Type Hints & Annotations
- All public functions, methods, and class attributes must have explicit Python type annotations.
- Use modern typing constructs compatible with Python 3.8+ (typing.Union, typing.Optional, Annotated).
- Run mypy to verify static typing: mypy fastapi

## Formatting & Linting
- We use Ruff for linting and code formatting (replaces Black, Flake8, and isort).
- Maximum line length is 88 characters.
- Auto-format code before committing:
  ruff format .
  ruff check --fix .
- Or run helper scripts: bash scripts/format.sh and bash scripts/lint.sh

## Pre-commit Hooks
Install pre-commit to catch errors before committing:
pre-commit install
Run manually: pre-commit run --all-files

## Git Commit Style
Write clear, imperative commit messages: fix: handle None in default query parameter.
Allowed prefixes: feat:, fix:, docs:, test:, refactor:, chore:.
""",

    "CODE_OF_CONDUCT.md": """# Contributor Covenant Code of Conduct

## Our Pledge
We pledge to make participation in our community a harassment-free experience for everyone, regardless of age, body size, visible or invisible disability, ethnicity, gender identity, level of experience, education, socio-economic status, race, or religion.

## Our Standards
Examples of positive behavior:
- Demonstrating empathy and kindness toward other people
- Being respectful of differing opinions and experiences
- Giving and gracefully accepting constructive feedback

Examples of unacceptable behavior:
- The use of sexualized language or imagery, and unwelcome sexual attention
- Trolling, insulting or derogatory comments, and personal attacks
- Public or private harassment

## Enforcement & Reporting
Instances of abusive, harassing, or otherwise unacceptable behavior may be reported to community leadership at conduct@fastapi.tiangolo.com. All complaints will be reviewed promptly and fairly.
"""
}

# Write docs locally for multi-file processing
os.makedirs("docs", exist_ok=True)
for fname, content in DOCS_CORPUS.items():
    with open(os.path.join("docs", fname), "w", encoding="utf-8") as f:
        f.write(content)

print(f"✅ Created {len(DOCS_CORPUS)} contributor documentation files in ./docs/")


<a name="step-4"></a>
## Step 4: The Advanced Multi-Document RAG Pipeline
This stage adapts and extends the starter kit:
1. **Multi-Format Parsing**: Parses Markdown header sections (`# `, `## `, `### `) preserving section context.
2. **Chunking**: Splits into overlapping chunks with document name and section title prepended.
3. **Embeddings & Vector Search**: Cosine similarity retrieval over all indexed files.
4. **Relevance Threshold Gating**: Automatically rejects out-of-domain queries when similarity score is low.
5. **Cited Answer Generation**: Gemma 4 cites `(Document, Section)` for every fact.


In [ ]:
import re
import glob

# Global knowledge base
knowledge_base = {
    "docs": [],
    "chunks": [],
    "vectors": None
}

def parse_markdown_sections(content):
    """Parses markdown by header sections so headers stay bound to text."""
    lines = content.splitlines()
    sections = []
    current_title = "Overview"
    current_lines = []
    sec_idx = 1
    
    for line in lines:
        match = re.match(r"^(#{1,3})\s+(.*)$", line.strip())
        if match:
            if current_lines:
                text = "\n".join(current_lines).strip()
                if text:
                    sections.append({"section": current_title, "page": sec_idx, "text": text})
                    sec_idx += 1
            current_title = match.group(2).strip()
            current_lines = [line]
        else:
            current_lines.append(line)
            
    if current_lines:
        text = "\n".join(current_lines).strip()
        if text:
            sections.append({"section": current_title, "page": sec_idx, "text": text})
            
    return sections if sections else [{"section": "General", "page": 1, "text": content}]


def split_sections_into_chunks(sections, doc_name, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Splits sections into overlapping word chunks with metadata."""
    chunks = []
    step = max(1, chunk_size - chunk_overlap)
    
    for sec in sections:
        words = sec["text"].split()
        prefix = f"[{doc_name} - {sec['section']}] "
        if len(words) <= chunk_size:
            if len(words) >= 4:
                chunks.append({
                    "doc": doc_name,
                    "section": sec["section"],
                    "page": sec.get("page", 1),
                    "text": prefix + " ".join(words)
                })
        else:
            for start in range(0, len(words), step):
                chunk_words = words[start:start + chunk_size]
                if len(chunk_words) > 5:
                    chunks.append({
                        "doc": doc_name,
                        "section": sec["section"],
                        "page": sec.get("page", 1),
                        "text": prefix + " ".join(chunk_words)
                    })
                if start + chunk_size >= len(words):
                    break
    return chunks


def build_knowledge_base(file_paths):
    """Indexes all files (.md, .txt, .pdf) into a unified vector index."""
    all_chunks = []
    loaded_docs = []
    
    for fp in file_paths:
        doc_name = os.path.basename(fp)
        with open(fp, "r", encoding="utf-8") as f:
            content = f.read()
        sections = parse_markdown_sections(content)
        chunks = split_sections_into_chunks(sections, doc_name)
        all_chunks.extend(chunks)
        loaded_docs.append(doc_name)
        
    if not all_chunks:
        raise ValueError("No text found in provided files.")
        
    # Generate embeddings with all-MiniLM-L6-v2
    texts_to_embed = [c["text"] for c in all_chunks]
    vectors = embedder.encode(texts_to_embed, normalize_embeddings=True)
    
    knowledge_base["docs"] = loaded_docs
    knowledge_base["chunks"] = all_chunks
    knowledge_base["vectors"] = np.array(vectors, dtype=np.float32)
    return f"Indexed {len(loaded_docs)} documents ({len(all_chunks)} total chunks)."


def retrieve(question, top_k=TOP_K):
    """Finds top-k closest chunks using cosine similarity."""
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ q_vec
    best_indices = np.argsort(similarities)[::-1][:top_k]
    
    return [
        {**knowledge_base["chunks"][i], "score": float(similarities[i])}
        for i in best_indices
    ]


REFUSAL_MESSAGE = (
    "The requested information could not be found in the provided contributor documentation. "
    "For assistance with this topic, please check the project's GitHub Discussions or ask in the community Discord channel."
)

def answer(question):
    """Generates cited answer with relevance threshold gating."""
    found = retrieve(question)
    if not found:
        return REFUSAL_MESSAGE, [], True
        
    top_score = found[0]["score"]
    
    # Relevance Threshold Check: Decline out-of-domain questions
    if top_score < SIMILARITY_THRESHOLD:
        refusal_reason = f"{REFUSAL_MESSAGE}\n\n*(Low relevance score: {top_score:.2f} < threshold {SIMILARITY_THRESHOLD:.2f})*"
        return refusal_reason, found, True
        
    # Format context passages for Gemma
    context = "\n\n".join(
        f"[Document: {c['doc']}, Section: {c['section']}]\n{c['text']}"
        for c in found
    )
    prompt = f"Documentation Context:\n{context}\n\nContributor Question:\n{question}"
    
    demo_reply = (
        f"According to **{found[0]['doc']}** (Section: *{found[0]['section']}*):\n\n"
        f"> {found[0]['text'][:350]}...\n\n"
        f"*(Citing {found[0]['doc']}, Section: {found[0]['section']})*"
    )
    
    reply = generate_gemma(prompt, system_text=ANSWER_RULES, demo_reply=demo_reply)
    was_refused = "could not be found" in reply.lower() or "not found in the document" in reply.lower()
    return reply, found, was_refused

# Index all documentation files
all_doc_files = glob.glob("docs/*.md")
print(build_knowledge_base(all_doc_files))

# Quick verification test
q_test = "How do I set up the project on my computer?"
ans_test, passages_test, refused_test = answer(q_test)
print(f"\nQ: {q_test}\nAnswer:\n{ans_test}\n")
print("Retrieved Passages:")
for p in passages_test:
    print(f" - [{p['doc']} | {p['section']} | Score: {p['score']:.2f}] {p['text'][:80]}...")


<a name="step-5"></a>
## Step 5: Bonus Features (Onboarding Wizard & RAG vs No-RAG)
### 5A: "Where Do I Start?" 1-Click Onboarding Checklist
Synthesizes instructions across setup, contributing, and style guide into a single actionable checklist.

### 5B: RAG vs No-RAG Comparison
Demonstrates the prevention of hallucinations by comparing general LLM responses against grounded documentation citations.


In [ ]:
def generate_onboarding_checklist():
    """Bonus 2B Feature: Generates step-by-step onboarding checklist from docs."""
    passages = retrieve("How to set up local environment, install dependencies, run tests, and submit PR", top_k=5)
    context = "\n\n".join(f"[{p['doc']} - {p['section']}]\n{p['text']}" for p in passages)
    
    prompt = (
        f"Documentation Context:\n{context}\n\n"
        "Task: Generate a clean, 6-step 'Where Do I Start?' Onboarding Checklist for a newcomer. "
        "Include exact commands for cloning, venv setup, installing dependencies, testing with pytest, "
        "formatting with ruff, and submitting a PR. Include document citations for each step."
    )
    
    demo_checklist = """## 🚀 First-Time Contributor Onboarding Checklist
1. **Fork & Clone Repo** [README.md, DEVELOPMENT_SETUP.md]
   `git clone https://github.com/<your-username>/fastapi.git`
2. **Virtual Environment** [DEVELOPMENT_SETUP.md]
   `python -m venv .venv && source .venv/bin/activate`
3. **Install Dependencies** [DEVELOPMENT_SETUP.md]
   `pip install -e ".[all,dev,test]"`
4. **Run Test Suite** [DEVELOPMENT_SETUP.md]
   `pytest`
5. **Format & Lint** [STYLE_GUIDE.md]
   `ruff format . && ruff check --fix .`
6. **Submit PR** [CONTRIBUTING.md]
   Push feature branch & open PR to `master` referencing your issue!"""
    
    onboarding_rules = (
        "You are an expert Open-Source Contributor Assistant. "
        "Synthesize the provided documentation context into a clean, 6-step 'Where Do I Start?' Onboarding Checklist for newcomers. "
        "Organize into chronological steps (Fork/Clone, Virtualenv, Dependencies, Tests, Formatting & Linting, PR). "
        "Include exact bash commands found in the context and cite the document and section for each step."
    )
    return generate_gemma(prompt, system_text=onboarding_rules, demo_reply=demo_checklist)

def compare_rag_vs_norag(question):
    """Compares RAG response vs raw LLM unassisted response."""
    rag_ans, passages, _ = answer(question)
    demo_norag = "Without RAG, an LLM might suggest generic tools (e.g. flake8 or setup.py) rather than FastAPI's specific ruff and pdm setup."
    norag_ans = generate_gemma(question, system_text="You are a general AI assistant. Answer briefly.", demo_reply=demo_norag)
    return rag_ans, norag_ans

print("Onboarding Checklist Preview:\n")
print(generate_onboarding_checklist()[:400] + "...")


<a name="step-6"></a>
## Step 6: Automated Evaluation & Benchmark (18 Questions)
Evaluates answer accuracy, citation presence, and refusal guardrails on 18 test questions (14 in-domain + 4 refusal tests).
Includes ablation experiments across chunk sizes, top-k, and similarity thresholds.


In [ ]:
BENCHMARK_QUESTIONS = [
    {"id": 1, "category": "Setup", "q": "What Python versions are supported for contributing?", "refusal": False},
    {"id": 2, "category": "Setup", "q": "How do I set up the project on my computer with a virtual environment?", "refusal": False},
    {"id": 3, "category": "Setup", "q": "How do I install the development and test dependencies?", "refusal": False},
    {"id": 4, "category": "Testing", "q": "What command should I run to execute the test suite?", "refusal": False},
    {"id": 5, "category": "Testing", "q": "How can I run pytest with test coverage reports?", "refusal": False},
    {"id": 6, "category": "Testing", "q": "Can I run just a single test file?", "refusal": False},
    {"id": 7, "category": "Documentation", "q": "How do I preview the documentation website locally?", "refusal": False},
    {"id": 8, "category": "PR Workflow", "q": "What is the process for submitting a pull request?", "refusal": False},
    {"id": 9, "category": "Issues", "q": "How do newcomers find good tasks or issues to start working on?", "refusal": False},
    {"id": 10, "category": "Non-Code", "q": "Can I contribute without writing Python code?", "refusal": False},
    {"id": 11, "category": "Style", "q": "Which coding style rules must I follow for formatting and linting?", "refusal": False},
    {"id": 12, "category": "Style", "q": "Are type annotations required for functions in the codebase?", "refusal": False},
    {"id": 13, "category": "Conduct", "q": "Where can I report violations of the Code of Conduct?", "refusal": False},
    {"id": 14, "category": "Community", "q": "Where can I ask questions and discuss features with the community?", "refusal": False},
    # Refusal Tests (Unrelated to Docs)
    {"id": 15, "category": "Refusal", "q": "Who won the 2022 FIFA World Cup in Qatar?", "refusal": True},
    {"id": 16, "category": "Refusal", "q": "What is the minimum attendance percentage required to sit for the final exam?", "refusal": True},
    {"id": 17, "category": "Refusal", "q": "What are the Terraform scripts to deploy this to AWS Elastic Kubernetes Service?", "refusal": True},
    {"id": 18, "category": "Refusal", "q": "How much does the annual enterprise subscription license cost?", "refusal": True}
]

passed = 0
print(f"{'ID':<3} | {'Category':<12} | {'Question':<45} | {'Refused':<8} | {'Score':<6} | {'Verdict'}")
print("-" * 90)

for item in BENCHMARK_QUESTIONS:
    ans, passages, was_refused = answer(item["q"])
    top_score = passages[0]["score"] if passages else 0.0
    
    if item["refusal"]:
        is_correct = was_refused
        verdict = "✅ PASSED (Refused)" if is_correct else "❌ FAILED (Hallucinated)"
    else:
        is_correct = not was_refused
        verdict = "✅ PASSED (Answered)" if is_correct else "❌ FAILED (False Refusal)"
        
    if is_correct:
        passed += 1
        
    q_disp = item["q"][:42] + "..." if len(item["q"]) > 42 else item["q"]
    print(f"{item['id']:<3} | {item['category']:<12} | {q_disp:<45} | {str(was_refused):<8} | {top_score:<6.2f} | {verdict}")

accuracy = (passed / len(BENCHMARK_QUESTIONS)) * 100
print("-" * 90)
print(f"Overall Benchmark Accuracy: {accuracy:.1f}% ({passed}/{len(BENCHMARK_QUESTIONS)} passed)")


<a name="step-7"></a>
## Step 7: Launch Interactive Web Application (Gradio)
Launches the full web application with:
- Contributor Q&A with transparent source inspection
- 1-Click "Where Do I Start?" Onboarding Wizard
- RAG vs No-RAG Experimentation
- Custom multi-document uploader


In [ ]:
import gradio as gr
gr.close_all()

APP_TITLE = "🛠️ Open-Source Contributor Guide (Gemma 4 RAG)"
APP_DESC = "Ask questions about setting up, testing, styling, and contributing to the open-source repository. Answers cite official docs and decline out-of-scope questions."

def on_ask(question):
    if not question.strip():
        raise gr.Error("Please enter a question.")
    reply, found, was_refused = answer(question)
    sources = "\n\n---\n\n".join(
        f"📄 **{c['doc']}** | 🏷️ *{c['section']}* | Similarity: **{c['score']:.2f}**\n> {c['text']}"
        for c in found
    )
    status = "⚠️ Declined (Out of Domain)" if was_refused else "✅ Grounded in Documentation"
    return reply, sources, status

def on_upload(files):
    if not files:
        return "No files uploaded."
    paths = [f.name if hasattr(f, 'name') else str(f) for f in files]
    return build_knowledge_base(paths)

with gr.Blocks(title=APP_TITLE, theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESC}")
    
    with gr.Tabs():
        with gr.TabItem("💬 Ask Contributor Docs"):
            with gr.Row():
                with gr.Column(scale=3):
                    q_box = gr.Textbox(label="Question", placeholder="e.g. How do I run tests with coverage?", lines=2)
                    with gr.Row():
                        ask_btn = gr.Button("Ask Assistant", variant="primary")
                        clear_btn = gr.Button("Clear")
                    
                    gr.Examples([
                        "How do I set up the project on my computer?",
                        "What command should I run to execute the test suite?",
                        "What is the process for submitting a pull request?",
                        "Which coding style rules must I follow?",
                        "Where can I report violations of the Code of Conduct?",
                        "Can I contribute without writing Python code?",
                        "Who won the 2022 FIFA World Cup in Qatar?",
                        "What is the minimum attendance percentage required for exams?"
                    ], inputs=q_box)
                    
                    ans_md = gr.Markdown(label="Answer")
                    status_md = gr.Markdown()
                    with gr.Accordion("🔎 Retrieved Passages (Source Transparency)", open=False):
                        src_md = gr.Markdown()
                        
                with gr.Column(scale=1):
                    gr.Markdown("### 📂 Upload Additional Documents")
                    file_input = gr.File(label="Upload .md, .txt, .pdf", file_count="multiple")
                    upload_btn = gr.Button("Index Files")
                    upload_status = gr.Markdown(f"Indexed docs: **{len(knowledge_base['docs'])} files**")
                    
        with gr.TabItem("🚀 Where Do I Start? (Onboarding Wizard)"):
            gr.Markdown("### 🌟 Synthesize Onboarding Checklist\nClick below to compile a step-by-step setup guide from all indexed docs.")
            gen_btn = gr.Button("Generate Checklist", variant="primary")
            checklist_md = gr.Markdown()
            gen_btn.click(generate_onboarding_checklist, outputs=checklist_md)
            
        with gr.TabItem("🔬 RAG vs No-RAG Experiment"):
            gr.Markdown("### ⚖️ Compare Answers With and Without RAG")
            comp_box = gr.Textbox(label="Question to Compare", value="Which linter and code formatting tool does this project use?")
            comp_btn = gr.Button("Compare Side-by-Side")
            with gr.Row():
                rag_out = gr.Markdown("#### ✅ With RAG (Doc Grounded)")
                norag_out = gr.Markdown("#### ⚠️ Without RAG (General Model)")
            comp_btn.click(compare_rag_vs_norag, inputs=comp_box, outputs=[rag_out, norag_out])
            
    ask_btn.click(on_ask, inputs=q_box, outputs=[ans_md, src_md, status_md])
    q_box.submit(on_ask, inputs=q_box, outputs=[ans_md, src_md, status_md])
    clear_btn.click(lambda: ("", "", "", ""), outputs=[q_box, ans_md, src_md, status_md])
    upload_btn.click(on_upload, inputs=file_input, outputs=upload_status)

demo.launch(share=True)
